In [2]:
# Install dependencies

!pip install -q transformers datasets peft accelerate bitsandbytes trl pyarrow evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 35.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 25.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 59.0 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.2

In [3]:
# Imports and Seeds

import os
import re
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [11]:
# Load and split dataset

dataset = load_dataset(
    "parquet",
    data_files="/kaggle/input/datasets/dhrvdng/hindi-toxic-dataset/hin-00000-of-00001.parquet"
)["train"]
 
dataset = dataset.shuffle(seed=SEED)
splits  = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = splits["train"]
test_dataset  = splits["test"]

train_dataset.to_parquet("/kaggle/working/train_split.parquet")
test_dataset.to_parquet("/kaggle/working/test_split.parquet")
 
print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")
 
# Check class balance
toxic_count     = sum(train_dataset["toxic"])
non_toxic_count = len(train_dataset) - toxic_count
print(f"Toxic: {toxic_count}  |  Non-toxic: {non_toxic_count}")

Generating train split: 0 examples [00:00, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Train: 3926  |  Test: 437
Toxic: 1975  |  Non-toxic: 1951


In [ ]:
# Load qwen tokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
 
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",   # right-pad for seq classification
)
 
# Qwen doesn't set a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token     = tokenizer.eos_token
    tokenizer.pad_token_id  = tokenizer.eos_token_id

In [12]:
# Tokenize

SYSTEM = "You are a Hindi/Hinglish toxicity classifier."
MAX_LEN = 256
 
def tokenize(batch):
    combined = [
        f"{SYSTEM}\nClassify: {t}"
        for t in batch["text"]
    ]
    enc = tokenizer(
        combined,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,          # DataCollatorWithPadding handles this
    )
    enc["labels"] = [int(x) for x in batch["toxic"]]
    return enc
 
train_tok = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
test_tok  = test_dataset.map(tokenize,  batched=True, remove_columns=test_dataset.column_names)
 
print("Sample keys:", train_tok[0].keys())
print("Input len sample:", len(train_tok[0]["input_ids"]))

Map:   0%|          | 0/3926 [00:00<?, ? examples/s]

Map:   0%|          | 0/437 [00:00<?, ? examples/s]

Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input len sample: 35


In [ ]:
# Load qwen as sequence classifier

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32,
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# Lora Config

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
 
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Class weight loss

n_total   = len(train_dataset)
n_toxic   = sum(train_dataset["toxic"])
n_nontox  = n_total - n_toxic
 
# Weight inversely proportional to frequency
weights = torch.tensor(
    [n_total / (2 * n_nontox), n_total / (2 * n_toxic)],
    dtype=torch.float32
).to(model.device)
 
print(f"Class weights — non-toxic: {weights[0]:.3f}, toxic: {weights[1]:.3f}")
 
class WeightedTrainer(Trainer):
    """Trainer subclass that applies class weights to cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Metrics

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1    = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    f1_tox = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    return {"accuracy": acc, "f1_weighted": f1, "f1_toxic": f1_tox}

In [ ]:
# Training

training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen-toxic",
    num_train_epochs=5,
    per_device_train_batch_size=8,          # seq cls uses less VRAM
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,          # effective batch = 32
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                       # warm up first 10 % of steps
    weight_decay=0.01,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch",
    dataloader_num_workers=2,
    seed=SEED,
)

In [ ]:
# Training

data_collator = DataCollatorWithPadding(tokenizer)
 
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
 
trainer.train()

In [ ]:
# Save model

SAVE_PATH = "/kaggle/working/qwen-toxic-final-new"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved to", SAVE_PATH)

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("hugging face")

# Push to HuggingFace as backup
from huggingface_hub import login
login(token=secret_value)
model.push_to_hub("dhrv-dng/qwen-toxic-classifier")
tokenizer.push_to_hub("dhrv-dng/qwen-toxic-classifier")

print("Saved locally + pushed to HuggingFace Hub")

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
import torch

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
login(token=secret.get_secret("hugging face"))


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
HF_REPO    = "dhrv-dng/qwen-toxic-classifier"

tokenizer = AutoTokenizer.from_pretrained(HF_REPO, trust_remote_code=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float32,
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

inf_model = PeftModel.from_pretrained(base_model, HF_REPO)

tokenizer_config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

In [5]:
def predict(text: str) -> int:
    """Return 1 if toxic, 0 if non-toxic."""
    prompt = f"{SYSTEM}\nClassify: {text}"
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(inf_model.device)
    with torch.no_grad():
        logits = inf_model(**enc).logits
    return int(logits.argmax(-1).item())

In [13]:
# Evaluation on test set

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
 
y_true, y_pred = [], []
 
for example in test_dataset:
    text  = example["text"]
    label = int(example["toxic"])
    pred  = predict(text)
    y_true.append(label)
    y_pred.append(pred)
 
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_true, y_pred):.4f}")
 
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))
 
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Accuracy  : 0.7529
Precision : 0.7671
Recall    : 0.7467
F1-score  : 0.7568

Classification Report:

              precision    recall  f1-score   support

           0     0.7385    0.7594    0.7488       212
           1     0.7671    0.7467    0.7568       225

    accuracy                         0.7529       437
   macro avg     0.7528    0.7531    0.7528       437
weighted avg     0.7533    0.7529    0.7529       437


Confusion Matrix:

[[161  51]
 [ 57 168]]


In [38]:
print(predict("bhai aaj mausam bada acha hai kahi bahar chalte hai ghumne")) 
print(predict("agar voh sun leta toh samajh jaata bhenchod"))

0
1


In [10]:
import pandas as pd

# Load your new dataset
new_df = pd.read_csv("/kaggle/input/datasets/dhrvdng/eval-lang/textdetox_hi_sample_168.csv")

y_true, y_pred = [], []

for _, row in new_df.iterrows():
    text  = row["text"]
    label = int(row["toxic"])
    pred  = predict(text)
    y_true.append(label)
    y_pred.append(pred)

print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_true, y_pred):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Accuracy  : 0.8393
Precision : 0.9672
Recall    : 0.7024
F1-score  : 0.8138

Classification Report:

              precision    recall  f1-score   support

           0     0.7664    0.9762    0.8586        84
           1     0.9672    0.7024    0.8138        84

    accuracy                         0.8393       168
   macro avg     0.8668    0.8393    0.8362       168
weighted avg     0.8668    0.8393    0.8362       168


Confusion Matrix:

[[82  2]
 [25 59]]
